# Clase 23: Reducción de dimensionalidad — PCA

## ¿Cómo comprimir muchas variables en pocas sin perder lo importante?

Imagina que tienes una escultura 3D y quieres mostrarla en una foto 2D. Puedes elegir el ángulo que revele más detalles o el que muestre menos. **PCA encuentra automáticamente el mejor ángulo** para proyectar tus datos de alta dimensión en 2 dimensiones, conservando la mayor cantidad de información posible.

In [ ]:
# Importar todas las librerías necesarias
# sklearn para PCA y escaling, matplotlib para visualizar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 11
print('Librerías cargadas ✅')

## 1. Cargar el dataset y explorar columnas numéricas

Usaremos el dataset de estudiantes que tiene varias columnas numéricas relacionadas con el rendimiento académico.

In [ ]:
# Cargar dataset de estudiantes
# Tiene múltiples columnas numéricas: notas, horas de estudio, faltas, etc.
import os

if os.path.exists('datasets/estudiantes.csv'):
    df = pd.read_csv('datasets/estudiantes.csv')
    print('Dataset real cargado.')
else:
    # Datos simulados con 5 variables correlacionadas (representan estudiantes)
    np.random.seed(42)
    n = 200
    base = np.random.randn(n)
    df = pd.DataFrame({
        'nota_matematicas': np.clip(60 + base * 15 + np.random.randn(n) * 5, 0, 100),
        'nota_ciencias':    np.clip(58 + base * 14 + np.random.randn(n) * 6, 0, 100),
        'nota_lengua':      np.clip(62 + base * 10 + np.random.randn(n) * 7, 0, 100),
        'horas_estudio':    np.clip(3 + base * 1.2 + np.random.randn(n) * 0.5, 0, 10),
        'faltas':           np.clip(5 - base * 2 + np.random.randn(n) * 1, 0, 20)
    })
    print('Datos simulados generados (5 variables correlacionadas).')

cols_num = df.select_dtypes(include='number').columns.tolist()
print(f'\nColumnas numéricas: {cols_num}')
print(f'Filas: {len(df)}')
df[cols_num].describe().round(2)

## 2. Matriz de correlación — ¿Qué variables se mueven juntas?

Antes de aplicar PCA, verificamos si hay variables correlacionadas. Variables muy correlacionadas son candidatas perfectas para comprimirse en una sola componente.

In [ ]:
# Visualizar la correlación entre todas las variables numéricas
# Valores cercanos a 1: correlación positiva fuerte
# Valores cercanos a -1: correlación negativa fuerte
# Valores cercanos a 0: sin correlación
X = df[cols_num].dropna()
corr_matrix = X.corr()

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr_matrix.values, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)

ax.set_xticks(range(len(cols_num)))
ax.set_yticks(range(len(cols_num)))
ax.set_xticklabels(cols_num, rotation=45, ha='right')
ax.set_yticklabels(cols_num)

for i in range(len(cols_num)):
    for j in range(len(cols_num)):
        ax.text(j, i, f'{corr_matrix.iloc[i,j]:.2f}',
                ha='center', va='center', fontsize=9)

ax.set_title('Matriz de correlación — Variables numéricas')
plt.tight_layout()
plt.show()

print('Variables con alta correlación son candidatas ideales para PCA')

## 3. Escalar los datos (paso obligatorio)

PCA calcula distancias. Sin escalar, las variables con valores grandes dominarían los resultados.

In [ ]:
# StandardScaler: lleva cada columna a media=0 y desv. est.=1
# Así todas las variables tienen igual 'peso' para PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Forma del array: {X_scaled.shape}')
print(f'Media por columna (debe ser ~0):    {X_scaled.mean(axis=0).round(4)}')
print(f'Desv. estándar (debe ser ~1): {X_scaled.std(axis=0).round(4)}')

## 4. Scree Plot — ¿Cuántos componentes necesito?

Primero analizamos todos los componentes para ver cuánta varianza captura cada uno. Buscamos el número mínimo que explique al menos el 80% de la información.

In [ ]:
# PCA completo: calcula todos los componentes posibles
# Esto nos permite ver cuánta información hay en cada dirección
pca_full = PCA()
pca_full.fit(X_scaled)

varianza_individual = pca_full.explained_variance_ratio_ * 100
varianza_acumulada  = varianza_individual.cumsum()
n_comp = len(varianza_individual)

print('Varianza explicada por componente:')
for i, (ind, acum) in enumerate(zip(varianza_individual, varianza_acumulada)):
    estrella = ' ← 80%' if abs(acum - 80) < 5 else ''
    print(f'  PC{i+1}: {ind:.1f}% (acumulada: {acum:.1f}%){estrella}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, n_comp+1), varianza_individual, color='steelblue', edgecolor='k')
axes[0].set_xlabel('Componente principal')
axes[0].set_ylabel('Varianza explicada (%)')
axes[0].set_title('Varianza por componente')

axes[1].plot(range(1, n_comp+1), varianza_acumulada, marker='o', color='coral')
axes[1].axhline(y=80, color='green', linestyle='--', alpha=0.7, label='80%')
axes[1].axhline(y=95, color='red', linestyle='--', alpha=0.7, label='95%')
axes[1].set_xlabel('Número de componentes')
axes[1].set_ylabel('Varianza acumulada (%)')
axes[1].set_title('Scree Plot — Varianza acumulada')
axes[1].legend()
axes[1].set_ylim(0, 105)

plt.tight_layout()
plt.show()

## 5. Aplicar PCA con 2 componentes

Aunque perdamos algo de información, reducir a 2 dimensiones nos permite visualizar todos los datos en un solo gráfico.

In [ ]:
# Reducir a 2 componentes para visualización
# fit_transform: calcula los componentes Y transforma los datos en un paso
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

var1 = pca.explained_variance_ratio_[0] * 100
var2 = pca.explained_variance_ratio_[1] * 100
var_total = var1 + var2

print(f'Forma original:  {X_scaled.shape}  ({X_scaled.shape[1]} variables)')
print(f'Forma reducida:  {X_pca.shape}      (2 componentes)')
print(f'\nPC1 explica: {var1:.1f}%')
print(f'PC2 explica: {var2:.1f}%')
print(f'Total conservado: {var_total:.1f}%  |  Perdido: {100-var_total:.1f}%')

## 6. Scatter plot 2D — Todos los estudiantes en un vistazo

In [ ]:
# Visualizar el dataset completo en 2 dimensiones
# Cada punto es un estudiante; puntos cercanos = estudiantes similares
plt.figure(figsize=(9, 6))
plt.scatter(
    X_pca[:, 0], X_pca[:, 1],
    alpha=0.6, edgecolors='k', linewidths=0.3, s=50, color='steelblue'
)
plt.axhline(0, color='gray', linewidth=0.5, linestyle='--')
plt.axvline(0, color='gray', linewidth=0.5, linestyle='--')
plt.xlabel(f'PC1 ({var1:.1f}% varianza)')
plt.ylabel(f'PC2 ({var2:.1f}% varianza)')
plt.title('Estudiantes proyectados en 2D con PCA')
plt.tight_layout()
plt.show()

print('¿Ves grupos o patrones en la nube de puntos?')

## 7. Biplot — ¿Qué miden los componentes principales?

El biplot agrega flechas que muestran cómo cada variable original contribuye a los componentes.

In [ ]:
# Biplot: puntos (muestras) + flechas (variables originales)
# Las flechas largas indican variables que contribuyen mucho
# Flechas en la misma dirección = variables correlacionadas
loadings = pca.components_.T  # (n_variables, 2)
escala = 2.0

fig, ax = plt.subplots(figsize=(9, 7))

# Puntos de los estudiantes
ax.scatter(X_pca[:, 0], X_pca[:, 1],
           alpha=0.35, s=30, color='#5C85D6', label='Estudiantes')

# Flechas de las variables
colores_flechas = ['#D32F2F', '#E64A19', '#F57C00', '#388E3C', '#1976D2']
for i, nombre in enumerate(cols_num):
    lx = loadings[i, 0] * escala
    ly = loadings[i, 1] * escala
    color = colores_flechas[i % len(colores_flechas)]
    ax.arrow(0, 0, lx, ly,
             head_width=0.07, head_length=0.04,
             fc=color, ec=color, linewidth=2)
    ax.text(lx * 1.15, ly * 1.15, nombre,
            fontsize=10, color=color, fontweight='bold', ha='center')

ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.set_xlabel(f'PC1 ({var1:.1f}% varianza)', fontsize=12)
ax.set_ylabel(f'PC2 ({var2:.1f}% varianza)', fontsize=12)
ax.set_title('Biplot PCA — Estudiantes', fontsize=13)
plt.tight_layout()
plt.show()

print('Interpretación de loadings:')
for i, nombre in enumerate(cols_num):
    print(f'  {nombre}: PC1={loadings[i,0]:.3f},  PC2={loadings[i,1]:.3f}')

## 8. PCA + K-Means: lo mejor de ambas clases

In [ ]:
# Combinar PCA con K-Means para clustering en el espacio reducido
# Ventaja: podemos visualizar los clusters directamente en 2D
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
etiquetas = kmeans.fit_predict(X_pca)

score = silhouette_score(X_pca, etiquetas)
print(f'Silhouette Score: {score:.3f}')

# Visualización
paleta = ['#2196F3', '#FF5722', '#4CAF50']
fig, ax = plt.subplots(figsize=(9, 6))

for i in range(3):
    mask = etiquetas == i
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=paleta[i], label=f'Cluster {i}',
               alpha=0.7, edgecolors='k', linewidths=0.3, s=55)

centros = kmeans.cluster_centers_
ax.scatter(centros[:, 0], centros[:, 1],
           marker='X', s=300, color='black',
           zorder=10, label='Centroides')

ax.set_xlabel(f'PC1 ({var1:.1f}%)')
ax.set_ylabel(f'PC2 ({var2:.1f}%)')
ax.set_title('K-Means en espacio PCA (3 clusters)')
ax.legend()
ax.axhline(0, color='gray', linewidth=0.4, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.4, linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
# Interpretar los clusters en las variables ORIGINALES (no en componentes PCA)
# Las componentes PCA no tienen nombre, pero las variables originales sí
df_resultado = X.copy()
df_resultado['cluster'] = etiquetas

perfil = df_resultado.groupby('cluster').mean().round(2)
conteo = df_resultado['cluster'].value_counts().sort_index().rename('n_estudiantes')

print('Perfil promedio de cada cluster (en variables originales):')
print(perfil.join(conteo))

print('\nInterpretación sugerida:')
print('  Cluster con notas altas y pocas faltas → Alto rendimiento')
print('  Cluster con notas bajas y muchas faltas → Bajo rendimiento')
print('  Cluster con valores medios → Rendimiento en desarrollo')